<a href="https://colab.research.google.com/github/willianbayer22/willianbayer22/blob/main/analise_selic_brasil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importão da biblioteca Pandas
import pandas as pd

# Importão da biblioteca Requests
import requests

In [2]:
# Endereço da API do Banco Central do Brasil.
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"

# Define o período que será analisado (2016 até 2025)
parametros = {
    "formato": "json",
    "dataInicial": "01/01/2016",
    "dataFinal": "31/12/2025"
}
resposta = requests.get(url, params=parametros)
print("Status da requisição:", resposta.status_code)

Status da requisição: 200


In [3]:
dados = resposta.json()
# Cria um DataFrame utilizando os dados obtidos do Banco Central.
df = pd.DataFrame(dados)
#Mostra os 5 primeiros resultados do dataFrame (usado pra teste)
df.head()

,data,valor
0,01/01/2016,14.25
1,02/01/2016,14.25
2,03/01/2016,14.25
3,04/01/2016,14.25
4,05/01/2016,14.25


In [4]:
#Veifica a quantidade de linhas e colunas do dataFrame
df.shape

(3653, 2)

In [5]:
# Exibe informações sobre as colunas do DataFrame,
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   data    3653 non-null   object
 1   valor   3653 non-null   object
dtypes: object(2)
memory usage: 57.2+ KB


In [6]:
# Verifica novamente se existem valores ausentes após a conversão dos tipos de dados.
df.isnull().sum()

,0
data,0
valor,0


In [7]:
# Apresenta estatísticas descritivas dos dados.
df.describe()

,data,valor
count,3653,3653
unique,3653,34
top,31/12/2025,6.50
freq,1,497


In [8]:
# Converte a coluna "data" do formato texto para o formato de data. O parâmetro dayfirst=True informa que o dia aparece antes do mês.
df["data"] = pd.to_datetime(df["data"], dayfirst=True)

# Mostra as informações atualizadas do DataFrame.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    3653 non-null   datetime64[ns]
 1   valor   3653 non-null   object        
dtypes: datetime64[ns](1), object(1)
memory usage: 57.2+ KB


In [9]:
# Converte a coluna "valor" de texto para número decimal.
# errors="coerce" transforma valores que não puderem ser convertidos em valores ausentes (NaN).
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")

# Mostra novamente as informações do DataFrame.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    3653 non-null   datetime64[ns]
 1   valor   3653 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 57.2 KB


In [10]:
# Cria uma nova coluna chamada "ano", extraindo o ano da coluna "data".
# Essa coluna será utilizada para realizar análises anuais.
df["ano"] = df["data"].dt.year

# Exibe os 5 primeiros registros para verificar o resultado.
df.head()

,data,valor,ano
0,2016-01-01,14.25,2016
1,2016-01-02,14.25,2016
2,2016-01-03,14.25,2016
3,2016-01-04,14.25,2016
4,2016-01-05,14.25,2016


In [12]:
# Encontra o maior valor registrado na coluna valor
maior_selic = df["valor"].max()

# Encontra o menor valor registrado na coluna valor
menor_selic = df["valor"].min()

# Exibe os resultados na tela
print("Maior taxa Selic:", maior_selic, "%")
print("Menor taxa Selic:", menor_selic, "%")

Maior taxa Selic: 15.0 %
Menor taxa Selic: 2.0 %


In [13]:
# Localiza os registros em que a Selic atingiu seu maior valor
maior_taxa = df.loc[df["valor"] == maior_selic, ["data", "valor"]]

# Localiza os registros em que a Selic atingiu seu menor valor
menor_taxa = df.loc[df["valor"] == menor_selic, ["data", "valor"]]

# Exibe os resultados na tela
print("Períodos com a maior taxa Selic:")
display(maior_taxa)

print("\nPeríodos com a menor taxa Selic:")
display(menor_taxa)

Períodos com a maior taxa Selic:


,data,valor
3457,2025-06-19,15.0
3458,2025-06-20,15.0
3459,2025-06-21,15.0
3460,2025-06-22,15.0
3461,2025-06-23,15.0
...,...,...
3648,2025-12-27,15.0
3649,2025-12-28,15.0
3650,2025-12-29,15.0
3651,2025-12-30,15.0



Períodos com a menor taxa Selic:


,data,valor
1679,2020-08-06,2.0
1680,2020-08-07,2.0
1681,2020-08-08,2.0
1682,2020-08-09,2.0
1683,2020-08-10,2.0
...,...,...
1898,2021-03-13,2.0
1899,2021-03-14,2.0
1900,2021-03-15,2.0
1901,2021-03-16,2.0


In [14]:
media_anual = df.groupby("ano")["valor"].mean()

# Exibe a média da Selic para cada ano.
print(media_anual)

ano
2016    14.178962
2017    10.154795
2018     6.580822
2019     6.035616
2020     2.886612
2021     4.537671
2022    12.547260
2023    13.296575
2024    10.940574
2025    14.417123
Name: valor, dtype: float64


In [15]:
# Identifica o ano que apresentou a maior média anual da Selic.
ano_maior_media = media_anual.idxmax()

# Obtém o valor da maior média anual.
maior_media = media_anual.max()

# Exibe os resultados
print("Ano com maior média da Selic:", ano_maior_media)
print("Média da Selic:", round(maior_media, 2), "%")

Ano com maior média da Selic: 2025
Média da Selic: 14.42 %


In [16]:
# Identifica o ano que apresentou a menor média anual da Selic.
ano_menor_media = media_anual.idxmin()

# Obtém o valor da menor média anual.
menor_media = media_anual.min()

# Exibe os resultados
print("Ano com menor média da Selic:", ano_menor_media)
print("Média da Selic:", round(menor_media, 2), "%")

Ano com menor média da Selic: 2020
Média da Selic: 2.89 %


In [17]:
# Exibe um resumo dos principais resultados encontrados na análise.

print("===== RESUMO DA ANÁLISE =====")
print(f"Período analisado: {df['data'].min().strftime('%d/%m/%Y')} até {df['data'].max().strftime('%d/%m/%Y')}")
print(f"Quantidade de registros: {len(df)}")
print(f"Maior taxa Selic: {maior_selic:.2f}%")
print(f"Menor taxa Selic: {menor_selic:.2f}%")
print(f"Ano com maior média: {ano_maior_media} ({maior_media:.2f}%)")
print(f"Ano com menor média: {ano_menor_media} ({menor_media:.2f}%)")

===== RESUMO DA ANÁLISE =====
Período analisado: 01/01/2016 até 31/12/2025
Quantidade de registros: 3653
Maior taxa Selic: 15.00%
Menor taxa Selic: 2.00%
Ano com maior média: 2025 (14.42%)
Ano com menor média: 2020 (2.89%)
